# LigandMPNN-FR Output Parser
Parses `stats/`, `seqs/`, and `relaxed/` directories from a pipeline run and visualizes per-cycle metrics.

In [ ]:
# ── Configuration ─────────────────────────────────────────────
out_folder = "../test_output/ligmpnn_fr"  # path to --out_folder used in the run
# ──────────────────────────────────────────────────────────────

In [ ]:
from glob import glob
import json
import os
import pandas as pd

def get_sequence(seqs_dir, cycle_tag, seq_idx):
    fa_path = f"{seqs_dir}/{cycle_tag}.fa"
    if not os.path.exists(fa_path):
        return None
    target_header = f">T={seq_idx}"
    with open(fa_path) as f:
        lines = f.readlines()
    capture = False
    for line in lines:
        if line.startswith(">"):
            capture = f"seq_{seq_idx}" in line or f"_seq_{seq_idx}" in line
            continue
        if capture and line.strip():
            return line.strip()
    return None

rows = []
for json_path in sorted(glob(f"{out_folder}/stats/*.json")):
    cycle_tag = os.path.basename(json_path).replace(".json", "")
    with open(json_path) as f:
        data = json.load(f)
    cycle_num = data.get("cycle", None)
    for seq_idx, score in enumerate(data.get("all_relax_scores", [])):
        rows.append({
            "cycle": cycle_num,
            "seq_idx": seq_idx,
            "mpnn": score.get("mpnn"),
            "ddg": score.get("metrics", {}).get("ddg"),
            "res_totalscore": score.get("metrics", {}).get("res_totalscore"),
            "cms": score.get("metrics", {}).get("cms"),
        })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} sequences across {df['cycle'].nunique()} cycles")
df.head()

In [ ]:
import matplotlib.pyplot as plt

metrics = ["mpnn", "ddg", "res_totalscore", "cms"]
per_cycle = df.groupby("cycle")[metrics].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, metric in zip(axes.flat, metrics):
    ax.plot(per_cycle["cycle"], per_cycle[metric], marker="o")
    ax.set_title(metric)
    ax.set_xlabel("Cycle")
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig("fastrelax_scores.svg")
plt.show()